> 📅 __Date: 2026-09-09__

# 📚 **Semi-Structured RAG**

> **Goal:** Understand how RAG can work with documents that contain both normal text and structured elements such as tables, while preserving the useful structure of those elements during retrieval and generation.

> **Prerequisite:** This chapter assumes familiarity with the previous RAG Architecture, RAG Implementation, Retriever, Embedding Models, and Text Splitter notes.

> **Dependencies:**

```python
!pip install -U langchain
!pip install -U langchain-classic
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-openai
!pip install -U langchain-chroma
!pip install -U chromadb
!pip install -U pypdf
!pip install -U "unstructured[all-docs]"
```

### **Install All Dependencies**

```python
!pip install -U langchain langchain-classic langchain-community langchain-text-splitters langchain-openai langchain-chroma chromadb pypdf "unstructured[all-docs]"
```

---

# 🧩 **What is Semi-Structured RAG?**

> **Semi-Structured RAG = A RAG approach that works with documents containing both unstructured text and structured elements such as tables.**

**A normal RAG pipeline usually works primarily with text:**

```text
Documents
   ↓
Text Extraction
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector DB
   ↓
Retriever
   ↓
LLM
   ↓
Answer
```

But real-world documents such as financial reports, annual reports, research papers, **and business documents often contain:**

```text
Text + Tables + Other Structured Elements
```

A standard text-only pipeline may lose important table relationships.

---

# 📊 **Text + Table**

**Many real-world PDFs contain both:**

```text
Text
+
Tables
```

**For example:**

```text
Tesla Annual Report

Revenue increased during the year...

---------------------------------------
| Year | Automotive Revenue | Total   |
|------|--------------------|---------|
| 2022 | ...                | ...     |
| 2023 | ...                | ...     |
| 2024 | ...                | ...     |
---------------------------------------
```

The surrounding text can explain the business context, while the table may contain the exact values needed to answer a question.

---

# ⚠️ **Challenges in Handling Tables for RAG**

## **1. Extraction**

> **The table structure should be preserved during extraction.**

**A PDF table should ideally retain relationships such as:**

```text
Column Header
     ↓
Row
     ↓
Cell Value
```

If extraction converts everything into plain text without preserving structure, important relationships can be lost.

---

## **2. Chunking**

> **Normal text chunking may break the logical structure of a table.**

For example:

```text
Original Table
        ↓
┌──────────────────────┐
│ Header               │
│ Row 1                │
│ Row 2                │
│ Row 3                │
└──────────────────────┘

       ↓ Chunking

Chunk 1 → Header + Row 1
Chunk 2 → Row 2
Chunk 3 → Row 3
```

Now the retrieved chunk may not contain enough information to understand the table correctly.

This motivates a more structure-aware approach.

---

# 📄 **Load the PDF**

For the examples in this chapter, we use a Tesla PDF.

```python
!pip install -U langchain-community pypdf
```

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/tsla (1).pdf")

pages = loader.load()

print(pages[10].page_content)

/tmp/ipykernel_4533/1461989526.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Automotive	leasing	revenue	decreased	$356	million,	or	14%,	in	the	year	ended	December	31,	2023	as	compared	to	the	year	ended	December	31,
2022.	The	decrease	was	primarily	due	to	a	decrease	in	direct	sales-type	leasing	revenue	driven	by	lower	deliveries	year	over	year,	partially	offset	by	an
increase	from	our	growing	direct	operating	lease	portfolio.
Services	and	other	revenue	increased	$2.23	billion,	or	37%,	in	the	year	ended	December	31,	2023	as	compared	to	the	year	ended	December	31,
2022.	The	increase	was	primarily	due	to	higher	used	vehicle	revenue	driven	by	increases	in	volume,	body	shop	and	part	sales	revenue,	non-warranty
maintenance	services	revenue,	paid	Supercharging	revenue	and	insurance	services	revenue,	all	of	which	are	primarily	attributable	to	our	growing	fleet.
The	increases	were	partially	offset	by	a	decrease	in	the	average	selling	price	of	used	vehicles.
Energy	Generation	and	Storage	Segment
Energy	generation	and	storage	revenue	includes	sales	and	leasing	of	solar	ene

The standard PDF loader extracts page content as text, but complex table structure may not be represented in a retrieval-friendly form.

---

# 🧱 **Unstructured**

> **Unstructured = A document-processing approach for making complex documents easier for downstream LLM applications.**

The Unstructured ecosystem can identify different document elements rather than treating the whole page as plain text.

**Typical elements can include:**

```text
Document
   ├── Title
   ├── NarrativeText
   ├── ListItem
   ├── Table
   ├── Image
   ├── Header
   ├── Footer
   └── Other Elements
```

### **Unstructured: Get Your Data LLM Ready**

**The following image illustrates common element types:**

<div align="center">
<img src="assets/unstructured.png" width="800" alt="Unstructured document element types">
<p><em>Figure: Common document elements identified during Unstructured processing.</em></p>
</div>

> **Note:** Exact element types depend on the document and the parsing strategy.

---

# 🔍 **How Unstructured Processes a PDF**

For complex PDFs, Unstructured can perform layout and element detection.

**A layout-detection model such as:**

```text
yolox_l0.05.onnx
```

can be used as part of the document-processing pipeline to help detect different page elements.

**The important idea is:**

```text
PDF Page
   ↓
Layout / Element Detection
   ↓
Text / Table / Image / Other Elements
```

---

# 📦 **Install Unstructured**

```python
!pip install -U unstructured[all-docs]
```

**For vector storage later in the notebook:**

```python
!pip install -U chromadb
```

> **Note:** Installing the full **`unstructured[all-docs]`** extra can pull in many dependencies. In a production environment, install only the integrations required for the document types you actually process.

---

# 🧪 **Partition the PDF**

In [2]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="/content/tsla (1).pdf",
    extract_images_in_pdf=False,
    infer_table_structure=True,
)

len(elements)

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

281

### **Important Parameters**

```text
extract_images_in_pdf=False
→ Do not extract images from the PDF

infer_table_structure=True
→ Attempt to preserve table structure
```

---

# 🔎 **Inspect the Extracted Elements**

In [3]:
elements[0].text

'UNITED STATES SECURITIES AND EXCHANGE COMMISSION'

Each element contains information about what was detected on the page.

**We can inspect the available element categories:**

In [4]:
set(ele.category for ele in elements)

{'Footer',
 'Header',
 'ListItem',
 'NarrativeText',
 'Table',
 'Title',
 'UncategorizedText'}

**Conceptually:**

```text
PDF
 ↓
Unstructured
 ↓
Elements
 ├── Text
 ├── Table
 ├── Image
 └── Other Elements
```

---

# 📊 **Extract Tables**

In [5]:
tables = [
    ele for ele in elements
    if ele.category == "Table"
]

**Inspect the text representation:**

```python
tables[0].text
```

**When table structure is inferred, the table may also provide an HTML representation:**

In [6]:
tables[3].metadata.text_as_html

'<table><thead><tr><th rowspan="2">Fremont Factory</th><th rowspan="2">Model S / Model X Model 3 / Model Y</th><th rowspan="2">Active Active</th></tr></thead><tbody><tr><td>Gigafactory Shanghai</td><td>Model 3 / Model Y</td><td>Active</td></tr><tr><td>Gigafactory Berlin-Brandenburg</td><td>Model Y</td><td>Active</td></tr><tr><td rowspan="2">Gigafactory Texas</td><td>Model Y</td><td>Active</td></tr><tr><td>Cybertruck</td><td>Active</td></tr><tr><td>Gigafactory Nevada</td><td>Tesla Semi</td><td>Pilot production</td></tr><tr><td>Various</td><td>Next Generation Platform</td><td>In development</td></tr></tbody></table>'

**Display it:**

In [7]:
from IPython.display import HTML

HTML(tables[3].metadata.text_as_html)

**This is useful because HTML can preserve relationships between:**

```text
Rows
Columns
Headers
Cells
```

better than a flattened plain-text representation.

---

# 🧠 **Why Represent Tables as HTML?**

**A table can be represented as:**

```html
<table>
    <tr>
        <th>Year</th>
        <th>Sales</th>
    </tr>
    <tr>
        <td>2023</td>
        <td>...</td>
    </tr>
</table>
```

This gives the downstream system explicit structural information.

**Conceptually:**

```text
Plain Text Table
→ Easier to flatten
→ Structure may become ambiguous

HTML Table
→ Rows / columns are explicit
→ Structure can be preserved
```

---

# 🏗️ **Semi-Structured RAG Architecture**

The key idea is to treat **text and tables differently during indexing**, while allowing them to participate in the same retrieval system.

```text
                         SOURCE DOCUMENT
                               │
                  ┌────────────┴────────────┐
                  ↓                         ↓
                 TEXT                     TABLE
                  │                         │
                  ↓                         ↓
              Chunking              HTML / Structured Form
                  │                         │
                  └────────────┬────────────┘
                               ↓
                         Create Summaries
                               ↓
                         Embedding Model
                               ↓
                           Vector DB
                               ↓
                            Retriever
                               ↓
                         Original Content
                               ↓
                              LLM
                               ↓
                            Answer
```

---

# 🗂️ **Indexing Strategy**

**A useful pattern for semi-structured RAG is:**

```text
Table (HTML) + ID
        ↓
       LLM
        ↓
 Table Summary + ID
        ↓
Embedding Model
        ↓
     Vectors
        ↓
    Vector DB
```

**At retrieval time:**

```text
User Query
    ↓
Retriever
    ↓
Table Summary + ID
    ↓
Original Table (HTML)
    ↓
LLM
    ↓
Response
```

**For normal text:**

```text
Text
 ↓
Embedding Model
 ↓
Vector DB
 ↓
Retriever
 ↓
LLM
 ↓
Response
```

> **Key idea:** Store a retrieval-friendly representation for search, while keeping the original content available for final answer generation.

---

# ✂️ **Chunking Unstructured Elements**

Unstructured elements can also be chunked.

In [8]:
from unstructured.chunking.basic import chunk_elements

chunks = chunk_elements(elements)

**Inspect the resulting categories:**

In [9]:
set(ele.category for ele in chunks)

{'CompositeElement', 'Table', 'TableChunk'}

**Conceptually:**

```text
Original Elements
       ↓
   Chunking
       ↓
Smaller Elements / Chunks
       ↓
 ┌─────┴─────┐
 ↓           ↓
Table       Text
```

---

# 🆔 **Why IDs Are Important**

When using summaries for retrieval, each summary needs a link back to its original content.

**For example:**

```text
table1 (id=a1)
   ↓
summary1 (id=a1)

table2 (id=b1)
   ↓
summary2 (id=b1)

table3 (id=c1)
   ↓
summary3 (id=c1)
```

**The ID acts as the bridge:**

```text
Summary
  ↓
Document ID
  ↓
Original Content
```

---

# 🔎 **Separate Tables and Text**

After chunking, separate tables from normal text.

In [10]:
table_elements = [
    ele for ele in chunks
    if ele.category in {"Table", "TableChunk"}
]

text_elements = [
    ele for ele in chunks
    if ele.category == "CompositeElement"
]

**Inspect text:**

In [11]:
text_elements[0].text

'UNITED STATES SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\nFORM 10-K\n\n(Mark One)\n\nx\n\nANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the fiscal year ended December 31, 2023\n\nOR\n\nfe)\n\nTRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from _________ to _________\n\nCommission File Number: 001-34756\n\nTesla, Inc.\n\n(Exact name of registrant as specified in its charter)\n\nDelaware'

**Inspect a table:**

In [12]:
table_elements[0].metadata.text_as_html

'<table/>'

---

# 📝 **Extract Text from Composite Elements**

In [13]:
text_data = [
    ele.text
    for ele in text_elements
]

---

# 📊 **Extract HTML Tables**

In [14]:
table_data = [
    ele.metadata.text_as_html
    for ele in table_elements
]

**Now we have:**

```text
text_data
→ Normal text chunks

table_data
→ Structured HTML table chunks
```

---

# 🤖 **Create Summaries**

The summaries act as the searchable representation.

**First install the OpenAI integration:**

```python
!pip install -U langchain-openai
```

**Set the API key in Colab:**

In [16]:
import os
from google.colab import userdata

openai = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai

**Create a prompt:**

In [17]:
from langchain_core.prompts import PromptTemplate

template = """
Your task is to create a detailed and concise summary of the given table or text.

Provide a clear summary that preserves the important facts, values, relationships,
and context needed for retrieval.

Chunk:
{chunk}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["chunk"]
)

**Create the model and chain:**

In [18]:
from langchain_openai import OpenAI

model = OpenAI()

summary_chain = prompt | model

**Test on one text chunk:**

In [19]:
summary_chain.invoke({
    "chunk": text_data[5]
})

'\nThis table provides information on the filing status of a registrant, indicating whether they are a large accelerated filer, accelerated filer, non-accelerated filer, smaller reporting company, or emerging growth company. These categories are defined in Rule 12b-2 of the Exchange Act.'

---

# 📝 **Generate Text Summaries**

**Instead of processing one chunk at a time:**

```python
# text_summaries = [
#     summary_chain.invoke({"chunk": text})
#     for text in text_data
# ]
```

**we can batch the requests:**

In [20]:
text_summaries = summary_chain.batch(text_data)

---

# 📊 **Generate Table Summaries**

**Use the same chain for tables:**

In [21]:
table_summaries = summary_chain.batch(table_data)

**Now we have:**

```text
Text Chunk  → Text Summary
Table       → Table Summary
```

---

# 🔀 **Multi-Vector Retriever**

> **MultiVectorRetriever = A retriever that stores searchable vector representations separately from the original documents, allowing multiple representations to point back to the same source content.**

This is particularly useful when we want to retrieve using summaries but return the original text or table.

---

# 🧠 **Multi-Vector Retrieval Architecture**

**For tables:**

```text
Original Table (id)
        ↓
       LLM
        ↓
   Table Summary (id)
        ↓
Embedding Model
        ↓
      Vectors
        ↓
    Vector DB
```

**And separately:**

```text
Original Table (id)
        ↓
   InMemoryStore
```

**At query time:**

```text
Query
  ↓
Retriever
  ↓
Table Summary
  ↓
Document ID
  ↓
Original Table
  ↓
LLM
  ↓
Response
```

**The same pattern can be used for normal text:**

```text
Original Text (id)
        ↓
       LLM
        ↓
    Summary (id)
        ↓
Embedding Model
        ↓
      Vector DB

Original Text (id)
        ↓
   InMemoryStore
```

---

# 🧰 **Create the MultiVectorRetriever**

In [22]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

vectordb = Chroma(
    collection_name="tesla",
    embedding_function=embedding_model,
    persist_directory="db"
)

docstore = InMemoryStore()

retriever = MultiVectorRetriever(
    vectorstore=vectordb,
    docstore=docstore,
    id_key="doc_id",
)

### **Storage Responsibility**

```text
Vector DB
→ Stores summary representations used for retrieval

Docstore
→ Stores original text / tables returned after retrieval
```

> **Important:** The vector store and document store play different roles.

---

# 🆔 **Generate Unique IDs**

In [23]:
import uuid

str(uuid.uuid4())

'd720dfc1-2cca-49d1-badf-59e1f5986ffa'

**For table summaries:**

In [24]:
table_ids = [
    str(uuid.uuid4())
    for _ in table_summaries
]

**For text summaries:**

In [25]:
text_ids = [
    str(uuid.uuid4())
    for _ in text_summaries
]

---

# 📄 **Create Summary Documents for Tables**

In [26]:
from langchain_core.documents import Document

table_summary_docs = []

for ind, summary in enumerate(table_summaries):

    doc = Document(
        page_content=summary,
        metadata={"doc_id": table_ids[ind]}
    )

    table_summary_docs.append(doc)

---

# 📝 **Create Summary Documents for Text**

In [27]:
text_summary_docs = []

for ind, summary in enumerate(text_summaries):

    doc = Document(
        page_content=summary,
        metadata={"doc_id": text_ids[ind]}
    )

    text_summary_docs.append(doc)

---

# 💾 **Store Summaries in Vector DB**

**The summaries are the searchable representations:**

In [28]:
retriever.vectorstore.add_documents(table_summary_docs)
retriever.vectorstore.add_documents(text_summary_docs)

['eaea19e6-b6fa-47d1-bc64-2f45583fd4e4',
 '4661220e-d967-499b-b7e1-2eedc1b24382',
 '18be3216-37a0-4df2-b951-ff822cd198c0',
 'b0e554e6-4382-4425-9598-3eeaa8e6fc86',
 '512ce0d3-8c1b-4a25-8a8c-df2f33b3173e',
 '3a4c83a2-bc55-4330-81cb-4c87fd682348',
 'c5d3d1ea-853e-4d27-b4c8-6f8bd78e2610',
 '090ccf74-d1af-4663-8309-4ec3ed39e9de',
 'b2374eca-fb02-457d-a9e4-0df2a376b730',
 '1f121412-e420-4cdf-97d2-0a88caceb821',
 'aa56282d-7039-4a4f-99d0-ba286c66ca54',
 'b0df3bce-af7b-435e-8fa4-d85cd86343a1',
 '2ea5ee7c-07dc-4b07-b407-98853abbea74',
 '76cac4f7-8c18-4282-ab75-3703248ac9f0',
 '9ecfd9da-5b15-45a5-acf3-ee99565e843a',
 '59bcfaf1-3d4b-499b-ad6f-0f5405edb027',
 '86d5c3c5-0059-494e-89b8-a4192ad4242c',
 '3da3ef8d-2a43-4f16-baa7-9a4a59d52e92',
 '6dd19d9c-1382-420a-baa2-b14c72a7da63',
 'e2967816-5aec-45d2-a9a0-1eaa638a1d25',
 '36420cc4-8e6e-40d6-8880-493a74460631',
 '6c38b1e9-56df-4bb1-9eb1-7fed9c59a504',
 '09f33ead-ccc3-481e-9344-57c670eacf1d',
 '248c9579-692f-49ba-a3fd-c62b56763880',
 '36549541-b1ff-

**Conceptually:**

```text
table_data  → table_ids
text_data   → text_ids
```

---

# 💾 **Store Original Data in InMemoryStore**

**Store the original tables and text using their corresponding IDs:**

In [29]:
retriever.docstore.mset(
    list(zip(table_ids, table_data))
)

retriever.docstore.mset(
    list(zip(text_ids, text_data))
)

**The relationship is:**

```text
Summary ID
    │
    ├── Vector DB → Summary
    │
    └── Docstore  → Original Content
```

---

# 🔍 **Retrieve Original Content**

**Now query the retriever:**

In [30]:
results = retriever.invoke(
    "How much was Tesla sales?"
)

results

['Automotive sales revenue increased $11.30 billion, or 17%, in the year ended December 31, 2023 as compared to the year ended December 31, 2022, primarily due to an increase of 473,382 combined Model 3 and Model Y cash deliveries from production ramping of Model Y globally. The increase was partially offset by a lower average selling price on our vehicles driven by overall price reductions year over year, sales mix, and a negative impact from the United States dollar strengthening against other',
 '<table><tr><td rowspan="2">(Dollars in millions) Automotive sales</td><td colspan="2"/><td colspan="2"/><td colspan="3"/><td colspan="2">$</td><td rowspan="2">% 17%</td><td colspan="2"/><td rowspan="2">% 52</td></tr><tr><td>$</td><td/><td/><td/><td/><td>44,125</td><td>$</td><td/><td>11,299</td><td/><td/></tr><tr><td>Automotive regulatory credits</td><td/><td/><td/><td/><td/><td>1,465</td><td/><td/><td>14</td><td>1%</td><td/><td/><td>21</td></tr><tr><td>Automotive leasing</td><td/><td/><td/>

**Conceptually:**

```text
Query
  ↓
Retriever
  ↓
Search Summary Vectors
  ↓
Find Matching ID
  ↓
Lookup Original Data
  ↓
Return Original Text / Table
```

This gives the LLM access to the original content instead of only the summary.

---

# 🏗️ **Complete Semi-Structured RAG Flow**

```text
                    DOCUMENT
                       │
             ┌─────────┴─────────┐
             ↓                   ↓
            TEXT               TABLE
             │                   │
             ↓                   ↓
          Chunking         Structured HTML
             │                   │
             └─────────┬─────────┘
                       ↓
                 Summarization
                       ↓
                Summary Documents
                       ↓
                 Embedding Model
                       ↓
                   Vector DB
                       ↓
                    Retriever
                       ↓
                   Document ID
                       ↓
                  Original Store
                       ↓
              Original Text / Table
                       ↓
                       LLM
                       ↓
                    Answer
```

---

# 🔗 **Build the RAG Chain**

**Create a prompt for the final answer:**

In [31]:
template = """
Based on the given context, answer the question.

If the answer is not present in the context, do not make up an answer.
Just say "I don't know".

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

model = OpenAI()

---

# 🔄 **RunnablePassthrough**

In [32]:
from langchain_core.runnables import RunnablePassthrough

> **RunnablePassthrough = Passes the input directly to the next component without changing it.**

**Conceptually:**

In [36]:
RunnablePassthrough().invoke("input")

'input'

**returns:**

```text
input
```

---

# 🔗 **Create the RAG Chain**

In [37]:
chain = {
    "context": retriever,
    "question": RunnablePassthrough()
} | prompt | model

**This means:**

```text
User Query
     │
     ├────────→ retriever → context
     │
     └────────→ RunnablePassthrough() → question
                              │
                              ↓
                         Prompt Template
                              ↓
                             LLM
                              ↓
                            Answer
```

---

# 🧠 **How the Dictionary-Based Chain Works**

```text
chain = {
    key1: component1,
    key2: component2
} | component3
```

**When:**

```python
chain.invoke("input")
```

**the execution is conceptually:**

```text
component1.invoke(input) → output1
component2.invoke(input) → output2

component3.invoke({
    key1: output1,
    key2: output2
})
```

---

# 🔗 **Sequential Chain**

**For a simple sequential chain:**

```text
Chain = Component1 | Component2 | Component3

chain.invoke("input")

1. output1 = component1.invoke(input)
2. output2 = component2.invoke(output1)
3. output3 = component3.invoke(output2)
```

**Final output:**

```text
output3
```

---

# 🧪 **Run the Semi-Structured RAG Chain**

In [35]:
chain.invoke(
    "How much was Tesla sales?"
)

"\nI don't know."

**Conceptually:**

```text
Question
   ↓
Retriever
   ↓
Summary Vector Search
   ↓
Matching Document ID
   ↓
Original Table / Text
   ↓
Context
   ↓
Prompt + Question
   ↓
LLM
   ↓
Answer
```

---

# 🆚 **Traditional RAG vs Semi-Structured RAG**

| Feature | Traditional RAG | Semi-Structured RAG |
|---|---|---|
| **Primary Data** | Text | Text + Tables |
| **Chunking** | Text chunks | Text chunks + structured table handling |
| **Table Structure** | May be flattened | Can be preserved |
| **Retrieval Representation** | Text / embeddings | Text and table summaries / embeddings |
| **Original Content** | Retrieved chunk | Original text or table |
| **Best For** | Text-heavy documents | Reports, financial documents, PDFs with tables |

---

# 🧠 **Why MultiVectorRetriever is Useful Here**

**The central problem is:**

```text
Search Representation
        ≠
Original Content
```

A summary can be excellent for retrieval while the original table is better for answering.

**Therefore:**

```text
Summary
→ Search

Original Table / Text
→ Answer
```

**This separation gives us:**

```text
Better Retrieval Representation
+
Original Detailed Context
```

---

# ⚖️ **Advantages**

```text
Preserves table structure
Handles text + tables together
Uses summaries for retrieval
Returns original content for generation
Can represent one source with multiple vectors
Useful for financial and business documents
```

---

# ⚠️ **Limitations**

```text
More complex pipeline
Additional summarization cost
Requires document IDs / bookkeeping
Table extraction quality depends on the parser
Large or complex tables may still require careful handling
More moving parts than basic text RAG
```

---

# 🧠 **Key Takeaways**

```text
Semi-Structured RAG
→ Text + Structured Elements

Unstructured
→ Detect and represent document elements

HTML Table
→ Preserve rows / columns / cells

Summary
→ Retrieval-friendly representation

MultiVectorRetriever
→ Search one representation, return another

Vector DB
→ Stores searchable summaries

Docstore
→ Stores original content

RAG
→ Retrieve original context → Generate answer
```

---

# 🧠 **Ultimate Memory Trick**

```text
TEXT
→ Chunk → Embed → Retrieve

TABLE
→ Structure → Summarize → Embed → Retrieve ID → Return Original

MULTIVECTOR
→ One Source → Multiple Representations

SEMI-STRUCTURED RAG
→ Text + Tables → Intelligent Retrieval
```

---

# 🎯 **Interview-Friendly Explanation**

> **Semi-Structured RAG is useful when documents contain both normal text and structured elements such as tables. Instead of flattening everything into plain text, we preserve table structure, create retrieval-friendly summaries, embed those summaries, and map the retrieved IDs back to the original table or text. This allows the retriever to search efficiently while the LLM receives the original detailed content for generating the final answer.**

---

# 🏁 **Final Mental Model**

```text
                    SEMI-STRUCTURED RAG
                             │
               ┌─────────────┴─────────────┐
               ↓                           ↓
             TEXT                        TABLE
               │                           │
               ↓                           ↓
           Chunking                   HTML / Structure
               │                           │
               └─────────────┬─────────────┘
                             ↓
                         SUMMARY
                             ↓
                      EMBEDDING MODEL
                             ↓
                         VECTOR DB
                             ↓
                          RETRIEVER
                             ↓
                        DOCUMENT ID
                             ↓
                       ORIGINAL STORE
                             ↓
                    ORIGINAL CONTENT
                       ┌─────┴─────┐
                       ↓           ↓
                     TEXT        TABLE
                       └─────┬─────┘
                             ↓
                            LLM
                             ↓
                           ANSWER
```

> **Remember:** Search the representation that is easiest to retrieve, but give the LLM the original information needed to answer accurately.